# Exploratory Data Analysis, Feature Engineering & Unsupervised Learning

## Project
**Mathematical Fingerprint of Musical Success**

## Purpose
This notebook focuses on exploring the cleaned datasets produced in ***math_music_data_cleaning.ipynb*** and preparing them for machine learning tasks.

The notebook covers:
- Exploratory Data Analysis (EDA)
- Feature Engineering
- Dimensionality Reduction (PCA)
- Clustering (K-Means)
- Time Series Analysis

---

## Data Setup (Required Before Running the Notebook)

### Step 1: Create the Data Folder
1. Open the **Files** panel (folder icon on the left side of Colab).
2. Create a folder named:

```text
data
```

### Step 2: Upload the Processed Datasets
Upload the following files generated in Notebook 1 into the `data/` folder:

```text
data/master_dataset_clean.csv
data/corgis_historical_clean.csv
```

### Step 3: Verify the Files
Run **CELL 1 – File Check** before continuing.

Expected structure:

```text
content/
│
├── data/
│   ├── master_dataset_clean.csv
│   └── corgis_historical_clean.csv
│
└── sample_data/
```

---

## Input Datasets

### master_dataset_clean.csv
Contains:
- Spotify audio features
- Billboard hit information
- Target variable (`is_hit`)

### corgis_historical_clean.csv
Contains:
- Historical music information
- Year-related variables
- Metadata used for time-series analysis

---

## Expected Outputs

By the end of this notebook we will produce:

- Correlation analysis
- Engineered musical features
- PCA components
- K-Means clusters
- Time-series visualizations
- Processed dataset for ***math_music_modeling_and_mlflow.ipynb***

---

In [ ]:
# ============================================================
# Topic: File Check
# Goal: Verify that files exist inside the 'data' folder
# ============================================================

import os

required_files = [
    "data/master_dataset_clean.csv",
    "data/corgis_historical_clean.csv"
]

all_ok = True
for f in required_files:
    if os.path.exists(f):
        size_mb = os.path.getsize(f) / (1024 * 1024)
        print(f"file: {f} — {size_mb:.2f} MB")
    else:
        print(f" {f} — NOT FOUND in 'data' folder.")
        all_ok = False

if all_ok:
    print("\n Validation successful. Continuing ...")

In [ ]:
# ============================================================
# Topic: Data Loading
# Goal: Load the cleaned datasets produced in math_music_data_cleaning.ipynb
# ============================================================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

# ------------------------------------------------------------
# Load cleaned datasets
# ------------------------------------------------------------

df = pd.read_csv("data/master_dataset_clean.csv")
df_hist = pd.read_csv("data/corgis_historical_clean.csv")

# ------------------------------------------------------------
# Audio features used throughout the project
# ------------------------------------------------------------

audio_features = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo"
]

print("Datasets loaded successfully")
print(f"Master dataset shape: {df.shape}")
print(f"Historical dataset shape: {df_hist.shape}")

display(df.head())

In [ ]:
# ============================================================
# Topic: Exploratory Data Analysis (EDA)
# Goal: Inspect dataset structure and available features
# ============================================================

print("=" * 60)
print("MASTER DATASET OVERVIEW")
print("=" * 60)

print("\nDataset shape:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
display(df.isnull().sum().sort_values(ascending=False).head(15))

print("\nData types:")
display(df.dtypes)

print("\nFirst statistics for numerical columns:")
display(df.describe())

In [ ]:
# ============================================================
# Topic: Exploratory Data Analysis (EDA) - Correlation Analysis
# Goal: Examine relationships between numerical audio features,
#       popularity, and hit status
# ============================================================

# ------------------------------------------------------------
# Select numerical columns for correlation analysis
# NOTE:
# Exclude 'peak_pos' and 'wks_on_chart' from the correlation
# matrix used for core interpretation because they come from
# Billboard and may introduce leakage in later modeling.
# ------------------------------------------------------------

corr_features = [
    "popularity",
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
    "is_hit"
]

corr_matrix = df[corr_features].corr()

print("Correlation matrix:")
display(corr_matrix)

# ------------------------------------------------------------
# Visualize correlation matrix
# ------------------------------------------------------------

plt.figure(figsize=(10, 8))
sns.heatmap(
    corr_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    linewidths=0.5
)

plt.title("Correlation Matrix of Audio Features, Popularity, and Hit Status")
plt.show()

### Observations: Correlation Analysis

Based on the generated Heatmap, several key mathematical and musical relationships can be identified:

*   **Energy and Loudness (0.76):** There is a strong positive correlation. This confirms that songs with higher physical volume (loudness) are mathematically perceived as more energetic.
*   **Energy and Acousticness (-0.73):** A strong negative correlation is observed. This suggests that as a song becomes more acoustic, its energy level significantly drops, reflecting the difference between electronic and acoustic instrumentation.
*   **Danceability and Valence (0.49):** There is a moderate positive relationship. This implies that "happier" or more positive songs (high valence) tend to be more danceable, which is a common trend in commercial music.
*   **Hit Status vs. Popularity (0.13):** The correlation is surprisingly low. This is a **non-trivial finding**: it suggests that being a "Billboard Hit" historically does not always translate to high "Popularity" on Spotify today, indicating a "popularity drift" over time.
*   **Instrumentalness and Loudness (-0.43):** A negative correlation, showing that instrumental tracks (songs without vocals) are generally mixed at lower loudness levels compared to vocal-heavy tracks.

---
**ML Observation:** The analysis shows that some features, such as Energy and Loudness, contain very similar information. To reduce redundancy and make the dataset easier to work with, we will apply Principal Component Analysis (PCA) in the next stage. This technique combines related features into a smaller set of variables while preserving most of the important information.

In [ ]:
# ============================================================
# Topic: Feature Engineering & Preprocessing
# Goal: Create new domain-specific features and scale numerical data
# ============================================================
from sklearn.preprocessing import StandardScaler

# creating an "Energy-to-Loudness Ratio"
# the strong mutual correlation suggests that analyzing deviations
# from the expected baseline is necessary for a deeper understanding of the data.
df['energy_loudness_ratio'] = df['energy'] / (np.abs(df['loudness']) + 1)

# "Mood Index" is created
# danceability and valence are combined due to their inherent relationship
df['mood_index'] = (df['danceability'] + df['valence']) / 2

# defining columns for scaling
# new signs are added to the analysis list
extended_features = audio_features + ['energy_loudness_ratio', 'mood_index']

# StandardScaler
# scaling is mandatory for the next step: PCA and Clustering
scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[extended_features] = scaler.fit_transform(df[extended_features])

print(f"Feature Engineering completed.")
print(f"New signs: energy_loudness_ratio, mood_index")
print(f"Data is scaled by StandardScaler.")
display(df_scaled[extended_features].head())

### Observations: Feature Engineering & Preprocessing

Two new domain-specific features were created based on the correlation analysis:

- **energy_loudness_ratio:** Captures the relationship between energy and loudness.
  Songs with high energy but low loudness will have a distinctly different ratio,
  potentially identifying "quiet but intense" tracks.

- **mood_index:** Combines danceability and valence into a single "mood" score.
  A high mood_index indicates songs that are both happy and danceable —
  a common characteristic of commercial hits.

All 11 features were adjusted to a standard range using **StandardScaler** (mean=0, std=1) so they can be compared fairly. This step is necessary because PCA and K-Means are influenced by the size of the numbers; without scaling, features with larger values would incorrectly dominate the results.

In [ ]:
# ============================================================
# Topic: Dimensionality Reduction (PCA)
# Goal: Reduce 11 features to 2D and 3D for visualization
#       and to understand the variance structure of the data
# ============================================================
from sklearn.decomposition import PCA

# ------------------------------------------------------------
# PCA with all components to analyze explained variance
# ------------------------------------------------------------
pca_full = PCA()
pca_full.fit(df_scaled[extended_features])

explained_variance = pca_full.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

# ------------------------------------------------------------
# Scree Plot (Elbow Plot for PCA)
# ------------------------------------------------------------
plt.figure(figsize=(10, 5))

plt.bar(range(1, len(explained_variance) + 1), explained_variance,
        alpha=0.7, label='Individual Explained Variance')
plt.plot(range(1, len(explained_variance) + 1), cumulative_variance,
         marker='o', color='red', label='Cumulative Explained Variance')

plt.axhline(y=0.80, color='gray', linestyle='--', label='80% threshold')

plt.xlabel('Principal Component')
plt.ylabel('Explained Variance Ratio')
plt.title('PCA: Explained Variance per Component')
plt.legend()
plt.tight_layout()
plt.show()

print("\nExplained variance per component:")
for i, v in enumerate(explained_variance):
    print(f"  PC{i+1}: {v:.4f} ({cumulative_variance[i]*100:.1f}% cumulative)")

# ------------------------------------------------------------
# PCA with 2 components for visualization
# ------------------------------------------------------------
pca_2d = PCA(n_components=2)
pca_result = pca_2d.fit_transform(df_scaled[extended_features])

df['PC1'] = pca_result[:, 0]
df['PC2'] = pca_result[:, 1]

# ------------------------------------------------------------
# Visualize PCA colored by Hit Status
# ------------------------------------------------------------
plt.figure(figsize=(10, 7))
sns.scatterplot(
    data=df.sample(3000, random_state=42),
    x='PC1', y='PC2',
    hue='is_hit',
    palette={0: 'steelblue', 1: 'crimson'},
    alpha=0.5,
    s=20
)
plt.title('PCA: 2D Projection of Songs colored by Hit Status')
plt.xlabel(f'PC1 ({explained_variance[0]*100:.1f}% variance)')
plt.ylabel(f'PC2 ({explained_variance[1]*100:.1f}% variance)')
plt.legend(title='Is Hit', labels=['Not a Hit', 'Hit'])
plt.tight_layout()
plt.show()

### Observations: Dimensionality Reduction (PCA)

- **Explained Variance:** The first 5 principal components explain approximately **81.4%** of the total variance in the dataset. This is a great result, as it allows us to reduce the feature space from 11 dimensions to 5 with minimal information loss.
- **Scree Plot:** We see a clear "elbow" around the 2nd and 3rd components. PC1 (34.1%) and PC2 (19.6%) capture over half of the data's characteristics.
- **2D Projection:** The scatter plot shows that "Hits" (in red) are not isolated in a single corner but are slightly more concentrated towards the center-right of the PC1 axis.
- **Non-trivial Finding:** The fact that hits and non-hits overlap significantly in 2D space suggests that musical success is **multi-dimensional**. Simple linear combinations of audio features are not enough to perfectly separate a hit from a non-hit, which justifies the use of more complex non-linear models (like Random Forest or XGBoost) in the next notebook.

In [ ]:
# ============================================================
# Topic: Clustering
# Goal: Identify natural groupings of songs using K-Means
#       and the Elbow Method
# ============================================================
from sklearn.cluster import KMeans

# Elbow Method to find optimal K
inertia = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(df_scaled[extended_features])
    inertia.append(kmeans.inertia_)

# plot Elbow
plt.figure(figsize=(10, 5))
plt.plot(K_range, inertia, marker='o', linestyle='--', color='darkgreen')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia (Within-cluster Sum of Squares)')
plt.title('Elbow Method for Optimal k')
plt.grid(True)
plt.show()

# applying K-Means with k=5 (often optimal for music: e.g., Relaxed, Energetic, Dance, Acoustic, etc.)
optimal_k = 5
kmeans_final = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df['cluster'] = kmeans_final.fit_predict(df_scaled[extended_features])

# visualize clusters in PCA Space
plt.figure(figsize=(10, 7))
sns.scatterplot(
    data=df.sample(3000, random_state=42),
    x='PC1', y='PC2',
    hue='cluster',
    palette='viridis',
    alpha=0.6,
    s=30
)
plt.title(f'K-Means Clustering (k={optimal_k}) projected on PCA axes')
plt.legend(title='Cluster')
plt.show()

### Observations: Clustering

- **Elbow Method:** The inertia graph shows a smooth decrease with a subtle "elbow" around **k=5**. This suggests that songs in our dataset can be naturally categorized into 5 different groups of songs based on their audio features.
- **Cluster Separation:** When projected onto the PC1/PC2 axes, the clusters show clear spatial boundaries.
    - **PC1 (Horizontal):** Likely separates songs by energy level (e.g., Acoustic vs. High-energy).
    - **PC2 (Vertical):** Likely separates songs by mood or complexity (e.g., Sad vs. Happy/Danceable).
- **Musical Interpretability:** For example, Cluster 2 (left) and Cluster 4 (top right) represent opposite musical profiles, capturing the diversity of the 81,000+ tracks.
- **Role in ML:** These cluster labels will be used as a **new engineered feature** in the next notebook. They tell the models which musical group the song belongs to.